In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, MaxNLocator
import numpy as np
import torch
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms

In [ ]:
def load_real_dataset(name: str, n: int) -> torch.Tensor:
    name = name.lower()
    if name == "mnist":
        dataset = torchvision.datasets.MNIST(
            root="./data",
            train=True,
            download=True,
            transform=transforms.Compose([transforms.ToTensor()]),
        )
    elif name == "cifar10":
        dataset = torchvision.datasets.CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform=transforms.Compose(
                [
                    transforms.ToTensor(),
                    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                ]
            ),
        )
    else:
        raise ValueError(f"Unknown dataset: {name}")

    loader = torch.utils.data.DataLoader(dataset, batch_size=n, shuffle=True)
    images, _ = next(iter(loader))
    return torch.flatten(images, 1)

In [ ]:
@dataclass
class ScheduleConfig:
    kind: str
    eta: float
    m: float = 0.0
    gamma: float = 0.99
    eta0: Optional[float] = None
    delta: Optional[float] = None
    eta_min: float = 0.0
    period: int = 200
    lr_sequence: Optional[np.ndarray] = None # Added for random schedule


def get_lr(cfg: ScheduleConfig, k: int) -> float:
    if k < 1:
        raise ValueError("k must be >= 1")

    if cfg.kind == "constant":
        return cfg.eta
    if cfg.kind == "poly":
        return cfg.eta / (k ** cfg.m)
    if cfg.kind == "exp":
        return cfg.eta * (cfg.gamma ** (k - 1))
    if cfg.kind == "linear":
        eta0 = cfg.eta0 if cfg.eta0 is not None else cfg.eta
        delta = cfg.delta if cfg.delta is not None else 0.0
        return max(eta0 - delta * (k - 1), 0.0)
    if cfg.kind == "cosine":
        if cfg.period <= 1:
            return cfg.eta
        phase = min(k - 1, cfg.period) / cfg.period
        return cfg.eta_min + 0.5 * (cfg.eta - cfg.eta_min) * (1.0 + math.cos(math.pi * phase))
    if cfg.kind == "random": # New random schedule
        if cfg.lr_sequence is None:
            raise ValueError("lr_sequence must be pre-generated for random schedule.")
        if k - 1 >= len(cfg.lr_sequence):
            # If k exceeds the pre-generated sequence length, return the last LR or 0
            return 0.0 # Or cfg.lr_sequence[-1] depending on desired behavior
        return cfg.lr_sequence[k - 1]

    raise ValueError(f"Unknown schedule kind: {cfg.kind}")


def cumulative_step_budget(cfg: ScheduleConfig, T: int) -> np.ndarray:
    vals = np.zeros(T + 1, dtype=float)
    for k in range(1, T + 1):
        vals[k] = vals[k - 1] + get_lr(cfg, k)
    return vals


def stopping_index_from_budget(target_budget: float, cfg: ScheduleConfig, T: int) -> int:
    budget = cumulative_step_budget(cfg, T)
    idx = np.searchsorted(budget, target_budget, side="left")
    return int(min(idx, T))

In [ ]:
def generate_eigen_data(alpha: float, p: int, n: int) -> Tuple[torch.Tensor, torch.Tensor]:
    Sigma = torch.zeros(p, p)
    for i in range(p):
        Sigma[i, i] = 1.0 / ((i + 1) ** (alpha / 2))
    Z = torch.randn(n, p) / np.sqrt(p)
    X = Z @ Sigma
    return X, Sigma.square()


def gradient_step(beta_k, X, y, lr, lam=0.0):
    n = X.shape[0]
    return beta_k - lr * ((X.T @ (X @ beta_k - y)) / n + lam * beta_k)


def compute_loss(beta, beta_star, Sigma):
    val = (beta - beta_star).T @ Sigma @ (beta - beta_star)
    return val.squeeze()


def perform_gd(X, y, T, schedule_cfg, beta_zero, beta_star, Sigma, lam=0.0):
    n, p = X.shape
    device = X.device # Get the device from the input tensor X
    betas = torch.zeros(p, T + 1, device=device) # Create on device
    losses = torch.zeros(T + 1, device=device) # Create on device

    beta = beta_zero.clone()
    for i in range(T + 1):
        losses[i] = compute_loss(beta, beta_star, Sigma)
        betas[:, [i]] = beta
        if i < T:
            lr = get_lr(schedule_cfg, i + 1)
            beta = gradient_step(beta, X, y, lr, lam=lam)

    return losses, betas


def mean_eigenvalue(X):
    n, p = X.shape
    eigvals = torch.linalg.svdvals(X).square()
    return eigvals.mean() if p < n else eigvals[:n].mean()


def median_eigenvalue(X):
    eigvals = torch.linalg.svdvals(X).square()
    return eigvals.median()


def min_eigenvalue(X):
    eigvals = torch.linalg.svdvals(X).square()
    return eigvals.min()


def compute_stopping_budget(tau, sigma, lbar, n):
    return n * np.log((sigma**2) / (tau**2) * lbar + 1.0) / lbar


def compute_stopping_budget_lam(tau, sigma, lbar, lam, n):
    numer = sigma**2 * lbar + tau**2
    denom = tau**2 - sigma**2 * lam * n
    if denom <= 0:
        return np.inf
    return np.log(numer / denom) / (lam + lbar / n)

In [ ]:
def apply_clean_axis_format(ax):
    ax.tick_params(axis="both", labelsize=12)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    yfmt = ScalarFormatter(useOffset=False)
    yfmt.set_scientific(False)
    ax.yaxis.set_major_formatter(yfmt)
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)


def plot_risk_panel(ax, losses, title, estimate_idx=None, true_idx=None, show_ylabel=False):
    xs = np.arange(len(losses))
    ax.plot(xs, losses, linewidth=2.5)
    if true_idx is not None:
        ax.axvline(true_idx, linewidth=2.0, linestyle="--", label="True optimum", color='green')
    if estimate_idx is not None and np.isfinite(estimate_idx):
        ax.axvline(estimate_idx, linewidth=2.0, label="Estimate", color='purple')

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Iteration", fontsize=13)
    if show_ylabel:
        ax.set_ylabel("Expected excess risk", fontsize=13)
    apply_clean_axis_format(ax)


def save_panel_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=300)
    plt.close(fig)

In [ ]:
def apply_clean_axis_format(ax):
    ax.tick_params(axis="both", labelsize=12)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    yfmt = ScalarFormatter(useOffset=False)
    yfmt.set_scientific(False)
    ax.yaxis.set_major_formatter(yfmt)
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)


def plot_risk_panel(ax, losses, title, estimate_idx=None, true_idx=None, show_ylabel=False):
    xs = np.arange(len(losses))
    ax.plot(xs, losses, linewidth=2.5)
    if estimate_idx is not None and np.isfinite(estimate_idx):
        ax.axvline(estimate_idx, linewidth=2.0, label="Estimate", color='purple')
    if true_idx is not None:
        ax.axvline(true_idx, linewidth=2.0, linestyle="--", label="True optimum", color='green')

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Iteration", fontsize=13)
    if show_ylabel:
        ax.set_ylabel("Expected excess risk", fontsize=13)
    apply_clean_axis_format(ax)


def save_panel_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=300)
    plt.close(fig)

def run_average_risk_experiment(
    X,
    Sigma,
    schedule_cfgs: List[ScheduleConfig],
    T: int,
    M: int,
    tau: float,
    sigma: float,
    lam: float = 0.0,
    lbar: Optional[float] = None,
    device: str = "cpu",
):
    n, p = X.shape
    beta_zero = torch.zeros(p, 1, device=device)
    lbar = float(mean_eigenvalue(X) if lbar is None else lbar)

    for cfg in schedule_cfgs:
        if cfg.kind == "random":
            if cfg.eta0 is None:
                raise ValueError("eta0 must be provided for random schedule.")
            cfg.lr_sequence = np.random.uniform(0, cfg.eta0 * 3, T)

    results = []
    for cfg in schedule_cfgs:
        loss_avg = torch.zeros(T + 1, device=device)
        mu_star = tau**2 / (n * sigma**2)

        if lam >= mu_star:
            budget_target = np.inf
            est_idx = None
        else:
            budget_target = (
                compute_stopping_budget_lam(tau, sigma, lbar, lam, n)
                if lam > 0
                else compute_stopping_budget(tau, sigma, lbar, n)
            )
            est_idx = stopping_index_from_budget(budget_target, cfg, T)

        for _ in tqdm(range(M), desc=f"{cfg.kind}"):
            beta_star = torch.randn(p, 1, device=device) / np.sqrt(p)
            epsilon = tau * torch.randn(n, 1, device=device)
            y = X @ beta_star + epsilon
            losses, _ = perform_gd(X, y, T, cfg, beta_zero, beta_star, Sigma, lam=lam)
            loss_avg += losses / M

        if lam >= mu_star:
            true_idx = None
        else:
            true_idx = int(loss_avg.argmin().item())

        results.append(
            {
                "cfg": cfg,
                "loss_avg": loss_avg.cpu().numpy(),
                "estimate_idx": est_idx,
                "true_idx": true_idx,
            }
        )
    return results


def figure_title_from_cfg(cfg: ScheduleConfig, lam: Optional[float] = None) -> str:
    base_title = ""
    if cfg.kind == "constant":
        base_title = r"Constant $\eta_k = \eta$"
    elif cfg.kind == "poly":
        base_title = r"Polynomial $\eta_k = \eta / k^{%s}$" % cfg.m
    elif cfg.kind == "exp":
        base_title = r"Exponential $\eta_k = \eta \cdot %s^{k-1}$" % cfg.gamma
    elif cfg.kind == "linear":
        base_title = r"Linear $\eta_k = \max(\eta - (k-1)\delta, 0)$"
    elif cfg.kind == "cosine":
        base_title = r"Cosine annealing"
    elif cfg.kind == "random":
        base_title = r"Random uniform $\eta_k \sim \mathcal{U}(0, 3\eta)$"
    else:
        base_title = cfg.kind

    if lam is not None:
        # Format lambda to at most 5 digits
        return base_title + r" ($\mu = %.5g$)" % lam
    return base_title


def make_schedule_comparison_figure(results, outfile, ncols=2, lam: Optional[float] = None):
    n_panels = len(results)
    nrows = math.ceil(n_panels / ncols)
    if nrows > 1:
      fig, axes = plt.subplots(nrows, ncols, figsize=(6.4 * ncols, 4.2 * nrows + 1), squeeze=False)
    else:
      fig, axes = plt.subplots(nrows, ncols, figsize=(6.4 * ncols, 4.2), squeeze=False)
    axes = axes.flatten()

    for i, result in enumerate(results):
        ax = axes[i]
        title = figure_title_from_cfg(result["cfg"], lam=lam)
        plot_risk_panel(
            ax,
            result["loss_avg"],
            title,
            estimate_idx=result["estimate_idx"],
            true_idx=result["true_idx"],
            show_ylabel=(i % ncols == 0),
        )

    for j in range(n_panels, len(axes)):
        axes[j].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=min(2, len(labels)), bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=12)

    plt.subplots_adjust(bottom=0.15)
    save_panel_figure(fig, outfile)

In [ ]:
# Real-data example (MNIST)

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_name = "mnist"
X = load_real_dataset(dataset_name, n=1000).to(device)
n, p = X.shape
Sigma = torch.eye(p, device=device)
lambda_max = float(torch.linalg.svdvals(X).max().square().item())
eta_base = (0.9*n) / (lambda_max)
tau = 1
sigma = 1.0 / np.sqrt(p)

mu_star = tau**2 / (n * sigma**2)
mu_values = [0.0, 0.5 * mu_star, 1.5 * mu_star]

for lam_val in mu_values:

  real_schedule_cfgs = [
      ScheduleConfig(kind="constant", eta=eta_base),
      ScheduleConfig(kind="poly", eta=eta_base, m=0.4),
      ScheduleConfig(kind="linear", eta=eta_base, eta0=eta_base, delta=eta_base / 300.0),
      ScheduleConfig(kind="exp", eta=eta_base, gamma=0.9825),
      ScheduleConfig(kind="cosine", eta=eta_base, eta_min=0.05 * eta_base, period=200),
      ScheduleConfig(kind="random", eta=eta_base, eta0=eta_base),
  ]

  real_results = run_average_risk_experiment(
      X=X,
      Sigma=Sigma,
      schedule_cfgs=real_schedule_cfgs,
      T=500,
      M=800,
      tau=tau,
      sigma=sigma,
      lam=lam_val,
      device=device,
  )

  # Constructing a comprehensive filename with all parameters
  output_filename = f"fig_real_{dataset_name}_n{n}_p{p}_tau{tau}_sig{sigma:.4g}_lmax{lambda_max:.4g}_eta{eta_base:.4g}_mu{lam_val:.5g}.pdf"
  make_schedule_comparison_figure(real_results, output_filename, ncols=2, lam=lam_val)
  print(f"Generated {output_filename}")

Using device: cuda


random: 100%|██████████| 800/800 [02:12<00:00,  6.06it/s]


Generated fig_real_mnist_n1000_p784_tau1_sig0.03571_lmax3.861e+04_eta0.02331_mu0.pdf


random: 100%|██████████| 800/800 [02:17<00:00,  5.84it/s]


Generated fig_real_mnist_n1000_p784_tau1_sig0.03571_lmax3.861e+04_eta0.02331_mu0.392.pdf


random: 100%|██████████| 800/800 [02:13<00:00,  5.97it/s]


Generated fig_real_mnist_n1000_p784_tau1_sig0.03571_lmax3.861e+04_eta0.02331_mu1.176.pdf


## Real-world Dataset Experiments

In [ ]:
# Real-data example (CIFAR10)

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset_name = "cifar10"
X = load_real_dataset(dataset_name, n=1000).to(device)
n, p = X.shape
Sigma = torch.eye(p, device=device)
lambda_max = float(torch.linalg.svdvals(X).max().square().item())
eta_base = (0.9*n) / (lambda_max)
tau = 1
sigma = 1.0 / np.sqrt(p)

mu_star = tau**2 / (n * sigma**2)
mu_values = [0.0, 0.5 * mu_star, 1.5 * mu_star]

for lam_val in mu_values:

  real_schedule_cfgs = [
      ScheduleConfig(kind="constant", eta=eta_base),
      ScheduleConfig(kind="poly", eta=eta_base, m=0.4),
      ScheduleConfig(kind="linear", eta=eta_base, eta0=eta_base, delta=eta_base / 300.0),
      ScheduleConfig(kind="exp", eta=eta_base, gamma=0.9825),
      ScheduleConfig(kind="cosine", eta=eta_base, eta_min=0.05 * eta_base, period=200),
      ScheduleConfig(kind="random", eta=eta_base, eta0=eta_base),
  ]

  real_results = run_average_risk_experiment(
      X=X,
      Sigma=Sigma,
      schedule_cfgs=real_schedule_cfgs,
      T=500,
      M=800,
      tau=tau,
      sigma=sigma,
      lam=lam_val,
      device=device,
  )

  # Constructing a comprehensive filename with all parameters
  output_filename = f"fig_real_{dataset_name}_n{n}_p{p}_tau{tau}_sig{sigma:.4g}_lmax{lambda_max:.4g}_eta{eta_base:.4g}_mu{lam_val:.5g}.pdf"
  make_schedule_comparison_figure(real_results, output_filename, ncols=2, lam=lam_val)
  print(f"Generated {output_filename}")

Using device: cuda


100%|██████████| 170M/170M [19:37<00:00, 145kB/s]
random: 100%|██████████| 800/800 [02:51<00:00,  4.67it/s]


Generated fig_real_cifar10_n1000_p3072_tau1_sig0.01804_lmax2.255e+05_eta0.003991_mu0.pdf


random: 100%|██████████| 800/800 [02:51<00:00,  4.65it/s]


Generated fig_real_cifar10_n1000_p3072_tau1_sig0.01804_lmax2.255e+05_eta0.003991_mu1.536.pdf


random: 100%|██████████| 800/800 [02:52<00:00,  4.63it/s]


Generated fig_real_cifar10_n1000_p3072_tau1_sig0.01804_lmax2.255e+05_eta0.003991_mu4.608.pdf


## Synthetic Eigen Dataset Experiments

Now, let's run a series of experiments using synthetic eigen data with different parameters for `p`, `n`, `tau`, and `alpha` to observe their effects on the learning schedules.

In [ ]:
# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define experiment configurations
eigen_experiment_configs = [
    {"p": 40, "n": 100, "tau": 1.0, "alpha": 2.0},
    {"p": 40, "n": 100, "tau": 1.0, "alpha": 4.0},
    {"p": 100, "n": 40, "tau": 0.15, "alpha": 2.0},
    {"p": 100, "n": 40, "tau": 0.15, "alpha": 4.0},
]

T_eigen = 200 # Number of iterations
M_eigen = 800 # Number of Monte Carlo samples

all_eigen_results = []

for i, cfg_params in enumerate(eigen_experiment_configs):
    p = cfg_params["p"]
    n = cfg_params["n"]
    tau = cfg_params["tau"]
    alpha = cfg_params["alpha"]

    print(f"\nRunning Eigen Experiment {i+1}: p={p}, n={n}, tau={tau}, alpha={alpha}")

    # Generate eigen data
    X_eigen, Sigma_eigen = generate_eigen_data(alpha=alpha, p=p, n=n)
    X_eigen = X_eigen.to(device)
    Sigma_eigen = Sigma_eigen.to(device)

    # Recalculate parameters based on new X_eigen and p
    lambda_max_eigen = float(torch.linalg.svdvals(X_eigen).max().square().item())
    eta_base_eigen = 0.9 / (lambda_max_eigen)
    sigma_eigen = 1.0 / np.sqrt(p)

    # Define schedule configurations for eigen experiments
    eigen_schedule_cfgs = [
        ScheduleConfig(kind="constant", eta=eta_base_eigen),
        ScheduleConfig(kind="poly", eta=eta_base_eigen, m=0.25),
        ScheduleConfig(kind="poly", eta=eta_base_eigen, m=0.5),
        ScheduleConfig(kind="linear", eta=eta_base_eigen, eta0=eta_base_eigen, delta=eta_base_eigen / 300.0),
        ScheduleConfig(kind="exp", eta=eta_base_eigen, gamma=0.995),
        ScheduleConfig(kind="cosine", eta=eta_base_eigen, eta_min=0.05 * eta_base_eigen, period=200),
        ScheduleConfig(kind="random", eta=eta_base_eigen, eta0=eta_base_eigen), # New random schedule
    ]

    # Run the experiment
    current_eigen_results = run_average_risk_experiment(
        X=X_eigen,
        Sigma=Sigma_eigen,
        schedule_cfgs=eigen_schedule_cfgs,
        T=T_eigen,
        M=M_eigen,
        tau=tau,
        sigma=sigma_eigen,
        lam=0.0,
        device=device,
    )
    all_eigen_results.append(current_eigen_results)

    # Make and save the comparison figure
    output_filename = f"figure_eigen_schedule_comparison_p{p}_n{n}_tau{tau}_alpha{alpha}.pdf"
    make_schedule_comparison_figure(current_eigen_results, output_filename, ncols=2)
    print(f"Generated {output_filename}")

print("All Eigen dataset experiments completed.")

Using device: cuda

Running Eigen Experiment 1: p=40, n=100, tau=1.0, alpha=2.0


random: 100%|██████████| 800/800 [00:50<00:00, 15.70it/s]


Generated figure_eigen_schedule_comparison_p40_n100_tau1.0_alpha2.0.pdf

Running Eigen Experiment 2: p=40, n=100, tau=1.0, alpha=4.0


random: 100%|██████████| 800/800 [00:50<00:00, 15.88it/s]


Generated figure_eigen_schedule_comparison_p40_n100_tau1.0_alpha4.0.pdf

Running Eigen Experiment 3: p=100, n=40, tau=0.15, alpha=2.0


random: 100%|██████████| 800/800 [00:51<00:00, 15.54it/s]


Generated figure_eigen_schedule_comparison_p100_n40_tau0.15_alpha2.0.pdf

Running Eigen Experiment 4: p=100, n=40, tau=0.15, alpha=4.0


random: 100%|██████████| 800/800 [00:51<00:00, 15.51it/s]


Generated figure_eigen_schedule_comparison_p100_n40_tau0.15_alpha4.0.pdf
All Eigen dataset experiments completed.


Figure 1

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# Check for CUDA availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define experiment configurations
cfg_params = {"p": 100, "n": 40, "tau": 0.35, "alpha": 2.0}

T_eigen = 200 # Number of iterations
M_eigen = 800 # Number of Monte Carlo samples

all_eigen_results = []

p = cfg_params["p"]
n = cfg_params["n"]
tau = cfg_params["tau"]
alpha = cfg_params["alpha"]

print(f"\nRunning Eigen Experiment: p={p}, n={n}, tau={tau}, alpha={alpha}")

# Generate eigen data
X_eigen, Sigma_eigen = generate_eigen_data(alpha=alpha, p=p, n=n)
X_eigen = X_eigen.to(device)
Sigma_eigen = Sigma_eigen.to(device)

# Recalculate parameters based on new X_eigen and p
lambda_max_eigen = float(torch.linalg.svdvals(X_eigen).max().square().item())
eta_base_eigen = (0.01*n) / (4 + lambda_max_eigen)
sigma_eigen = 1.0 / np.sqrt(p)

# Define schedule configurations for eigen experiments
eigen_schedule_cfgs = [
    ScheduleConfig(kind="constant", eta=eta_base_eigen),
]

# Run the experiment
for lambd in [0, tau**2 * 0.5, tau**2 * 2.5, tau**2 * 4]:
  current_eigen_results = run_average_risk_experiment(
      X=X_eigen,
      Sigma=Sigma_eigen,
      schedule_cfgs=eigen_schedule_cfgs,
      T=T_eigen,
      M=M_eigen,
      tau=tau,
      sigma=sigma_eigen,
      lam=lambd,
      device=device,
  )
  all_eigen_results.append(current_eigen_results)

  # Make and save the comparison figure
  output_filename = f"figure_eigen_schedule_comparison_p{p}_n{n}_tau{tau}_alpha{alpha}_mu{lambd}.pdf"
  make_schedule_comparison_figure(current_eigen_results, output_filename, ncols=1, lam=lambd)
  print(f"Generated {output_filename}")

print("All Eigen dataset experiments completed.")

Using device: cuda

Running Eigen Experiment: p=100, n=40, tau=0.35, alpha=2.0


constant: 100%|██████████| 800/800 [00:50<00:00, 15.72it/s]


Generated figure_eigen_schedule_comparison_p100_n40_tau0.35_alpha2.0_mu0.pdf


constant: 100%|██████████| 800/800 [00:51<00:00, 15.68it/s]


Generated figure_eigen_schedule_comparison_p100_n40_tau0.35_alpha2.0_mu0.06124999999999999.pdf


constant: 100%|██████████| 800/800 [00:49<00:00, 16.20it/s]


Generated figure_eigen_schedule_comparison_p100_n40_tau0.35_alpha2.0_mu0.30624999999999997.pdf


constant: 100%|██████████| 800/800 [00:48<00:00, 16.41it/s]


Generated figure_eigen_schedule_comparison_p100_n40_tau0.35_alpha2.0_mu0.48999999999999994.pdf
All Eigen dataset experiments completed.
